## Import librairies

In [768]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import plotly.express as px
from python_module.pricing_model import BSMModel

pd.options.display.max_rows = 999
pd.options.display.max_columns = 999
pd.options.display.float_format = '{:,.2f}'.format

In [769]:
def compute_backtest(days_to_maturity, strike_pct, price_ts, option_type, sigma):
    price_ts.name = 'F'
    # build cyclic count 5->0 and, after each 0, insert a duplicated row that starts the next cycle at 5
    orig = price_ts.reset_index()
    idx_col = orig.columns[0]  # original index column name (usually the timestamp)
    rows = []
    count = days_to_maturity

    for _, r in orig.iterrows():
        d = r.to_dict()
        d['days_to_maturity'] = count
        rows.append(d)
        if count == 0:
            # duplicate the same row but set count to 5 to start the next cycle
            dup = r.to_dict()
            dup['days_to_maturity'] = days_to_maturity
            rows.append(dup)
            count = days_to_maturity-1  # next appended original row should get 4
        else:
            count -= 1

    bt_df = pd.DataFrame(rows).set_index(idx_col)

    bt_df.loc[bt_df['days_to_maturity']==days_to_maturity, 'F0'] = bt_df.loc[bt_df['days_to_maturity']==days_to_maturity, 'F']
    bt_df.loc[bt_df['days_to_maturity']==days_to_maturity, 'strike_date'] = bt_df.loc[bt_df['days_to_maturity']==days_to_maturity].index
    bt_df = bt_df.ffill()
    bt_df['K'] = bt_df['F0'] * strike_pct
    bt_df['T'] = bt_df['days_to_maturity'] / 252

    rows = []
    for index, row in bt_df.iterrows():
        
        row_dict = row.to_dict()

        row_dict['date'] = index
        F = row_dict['F']
        K = row_dict['K']
        T = row_dict['T']

        pricing_results = BSMModel.compute_option_with_forward(
            F=F,
            K=K,
            T=T,
            r=0,
            sigma=sigma,
            option_type=option_type,
            compute_greeks=True
            )
        merged_dict = {**row_dict, **pricing_results}
        rows.append(merged_dict)
    bt_df = pd.DataFrame(rows)
    bt_df['dP'] = bt_df['price'].diff()
    bt_df['dH'] = bt_df['F'].diff() * bt_df['delta'].shift(1)

    bt_df.loc[bt_df['days_to_maturity']==days_to_maturity, 'dP'] = 0 
    bt_df.loc[bt_df['days_to_maturity']==days_to_maturity, 'dH'] = 0

    # Extra outputs
    bt_df['dP_cumsum'] = bt_df['dP'].cumsum()
    bt_df['dH_cumsum'] = bt_df['dH'].cumsum()
    return bt_df

## Inputs

In [770]:
long_asset_symbol = 'QQQ'
long_asset_vol = 0.10

short_asset_symbol = 'SPY'
short_asset_vol = 0.10

vol_window = 20
beta_window = 5

option_type = 'call'
target_delta = 0.1
target_slide = 0.3
slide_list = [target_slide, 0.2, 0.1, 0.05]

beta_type = 'quantile' # 'mean', 'quantile' or 'override'
override_value = 1
quantile = 10

## Import data

In [771]:
long_assset_df = pd.read_csv(f'data/{long_asset_symbol}.csv', index_col=0, parse_dates=True)['price']
short_asset_df = pd.read_csv(f'data/{short_asset_symbol}.csv', index_col=0, parse_dates=True)['price']
long_assset_df.name = long_asset_symbol
short_asset_df.name = short_asset_symbol
df = pd.concat([long_assset_df, short_asset_df], axis=1)

## Volatility Target

In [772]:
vol_target = pd.Series(index=[long_asset_symbol, short_asset_symbol], data=[long_asset_vol, short_asset_vol], name='target_vol')
df_log_returns = np.log(df / df.shift(1))
df_rolling_std = df_log_returns.rolling(window=vol_window).std() * np.sqrt(252)
df_leverage = vol_target / df_rolling_std
df_leverage = df_leverage.dropna()
df = df.loc[df_leverage.index]
df_vt = (df.pct_change() * df_leverage.shift(1)).fillna(0).add(1).cumprod() * 100
df_vt_change = df_vt.pct_change()
df_vt_level = df_vt_change.fillna(0).add(1).cumprod() * 100

## Beta calibration

In [773]:
df_vt_roll_returns = df_vt_level.rolling(window=beta_window).apply(lambda x: x.iloc[-1]/x.iloc[0]-1).dropna()
beta_ts = df_vt_roll_returns[long_asset_symbol] / df_vt_roll_returns[short_asset_symbol]

df_regression = df_vt_roll_returns[[long_asset_symbol, short_asset_symbol]].copy()
df_regression['beta'] = beta_ts

long_asset_K = BSMModel.solve_delta_strike(F=100, T=beta_window/252, sigma=long_asset_vol, r=0, option_type=option_type, target_delta=target_delta)
short_asset_K = BSMModel.solve_delta_strike(F=100, T=beta_window/252, sigma=short_asset_vol, r=0, option_type=option_type, target_delta=target_delta)
short_asset_threshold = short_asset_K / 100 -1

if option_type == 'put':
    cond_serie = df_regression[df_regression[short_asset_symbol] < short_asset_threshold]['beta']
else:
    cond_serie = df_regression[df_regression[short_asset_symbol] > short_asset_threshold]['beta']

if beta_type == 'mean':
    cond_beta = cond_serie.mean()
elif beta_type == 'quantile':
    cond_beta = np.percentile(cond_serie, quantile)
elif beta_type == 'override':
    cond_beta = override_value

print(f'Beta: {cond_beta:.2f}')

Beta: 0.51


## Backtesting

In [774]:
pricing = dict()

beta_slide = np.round(target_slide*cond_beta, 2)
slide_list += [beta_slide]

long_asset_out = BSMModel.compute_option(F=100, K=long_asset_K, T=beta_window/252, r=0, sigma=long_asset_vol, option_type=option_type, compute_greeks=True, slide_list=slide_list)
short_asset_out = BSMModel.compute_option(F=100, K=short_asset_K, T=beta_window/252, r=0, sigma=short_asset_vol, option_type=option_type, compute_greeks=True, slide_list=slide_list)

long_asset_out['moneyness'] = long_asset_K / 100
short_asset_out['moneyness'] = short_asset_K / 100

long_asset_out['target_slide'] = long_asset_out[beta_slide]
short_asset_out['target_slide'] = short_asset_out[target_slide]

pricing[long_asset_symbol] = long_asset_out
pricing[short_asset_symbol] = short_asset_out

pricing_df = pd.DataFrame(pricing).transpose()
pricing_df['qty'] = 100_000_000 / pricing_df['target_slide']
pricing_df['delta_cash'] = pricing_df['delta'] * 100
for key in ['price', 'delta_cash', 'gamma', 'vega', 'theta'] + slide_list:
    pricing_df[f'ccy_{key}'] = pricing_df[key] * pricing_df['qty']
cols = list(filter(lambda x: str(x).startswith('ccy'), pricing_df.columns))
display(pricing_df[cols])
display(pricing_df[cols].multiply(pd.Series(index=[long_asset_symbol, short_asset_symbol], data=[1,-1]), axis=0).sum())

,ccy_price,ccy_delta_cash,ccy_gamma,ccy_vega,ccy_theta,ccy_0.3,ccy_0.2,ccy_0.1,ccy_0.05,ccy_0.15
QQQ,"506,196.79","76,345,862.44","951,126.60","188,715.60","-188,715.60","214,484,161.72","138,161,387.24","61,838,612.80","23,735,485.81","100,000,000.00"
SPY,"236,006.61","35,595,104.94","443,448.41","87,985.80","-87,985.80","100,000,000.00","64,415,659.47","28,831,318.97","11,066,311.67","46,623,489.21"


ccy_price            270,190.19
ccy_delta_cash    40,750,757.50
ccy_gamma            507,678.19
ccy_vega             100,729.80
ccy_theta           -100,729.80
ccy_0.3          114,484,161.72
ccy_0.2           73,745,727.76
ccy_0.1           33,007,293.83
ccy_0.05          12,669,174.14
ccy_0.15          53,376,510.79
dtype: float64

In [775]:
long_asset_payoff = df_vt_level[long_asset_symbol].rolling(window=beta_window).apply(lambda x: x.iloc[-1]/x.iloc[0]-1)
long_asset_payoff = long_asset_payoff.to_frame(name='returns')
long_asset_payoff['ST'] = 100 * (1+long_asset_payoff['returns'])
long_asset_payoff['K'] = long_asset_K
if option_type == 'call':
    long_asset_payoff['payoff'] = (long_asset_payoff['ST']-long_asset_payoff['K']).clip(lower=0)
else:
    long_asset_payoff['payoff'] = (long_asset_payoff['K']-long_asset_payoff['ST']).clip(lower=0)
long_asset_payoff['total_pnl'] = long_asset_payoff['payoff'] - pricing_df.loc[long_asset_symbol, 'price']
long_asset_payoff['ccy_total_pnl'] = long_asset_payoff['total_pnl'] * pricing_df.loc[long_asset_symbol, 'qty']

short_asset_payoff = df_vt_level[short_asset_symbol].rolling(window=beta_window).apply(lambda x: x.iloc[-1]/x.iloc[0]-1)
short_asset_payoff = short_asset_payoff.to_frame(name='returns')
short_asset_payoff['ST'] = 100 * (1+short_asset_payoff['returns'])
short_asset_payoff['K'] = short_asset_K
if option_type == 'call':
    short_asset_payoff['payoff'] = (short_asset_payoff['ST']-short_asset_payoff['K']).clip(lower=0)
else:
    short_asset_payoff['payoff'] = (short_asset_payoff['K']-short_asset_payoff['ST']).clip(lower=0)

short_asset_payoff['total_pnl'] = short_asset_payoff['payoff'] - pricing_df.loc[short_asset_symbol, 'price']
short_asset_payoff['ccy_total_pnl'] = short_asset_payoff['total_pnl'] * pricing_df.loc[short_asset_symbol, 'qty']

x = (long_asset_payoff-short_asset_payoff)['ccy_total_pnl'].fillna(0)

## Plot

In [776]:
px.bar(x.groupby(x.index.year).sum())

In [777]:
x_cumsum = x.cumsum()
fig = px.line(x_cumsum)
for year in x.index.year.unique():
    fig.add_vline(x=pd.Timestamp(year=year, month=1, day=1), line_dash="dot", line_color="red")
fig.show()

In [778]:
long_bt = compute_backtest(
    days_to_maturity=beta_window, 
    strike_pct=pricing_df.loc[long_asset_symbol, 'moneyness'], 
    price_ts=df_vt_level[long_asset_symbol].copy(), 
    option_type=option_type, 
    sigma=long_asset_vol)

short_bt = compute_backtest(
    days_to_maturity=beta_window, 
    strike_pct=pricing_df.loc[short_asset_symbol, 'moneyness'], 
    price_ts=df_vt_level[short_asset_symbol].copy(), 
    option_type=option_type, 
    sigma=short_asset_vol)

long_bt = long_bt.groupby('strike_date')['dH'].sum() * (100 / long_bt.groupby('strike_date')['F0'].mean())
short_bt = short_bt.groupby('strike_date')['dH'].sum() * (100 / short_bt.groupby('strike_date')['F0'].mean())

In [779]:
long_bt = long_bt.cumsum()
short_bt = short_bt.cumsum()

In [780]:
x = long_bt * pricing_df.loc[long_asset_symbol, 'qty'] - short_bt * pricing_df.loc[short_asset_symbol, 'qty']

In [781]:
px.line(x)

In [782]:
px.scatter(cond_serie)

In [783]:
df_regression.sort_values(by=short_asset_symbol).head(10)

,QQQ,SPY,beta
2015-08-24,-0.07,-0.08,0.97
2015-08-25,-0.07,-0.07,0.95
2007-02-27,-0.04,-0.07,0.63
2018-10-11,-0.04,-0.07,0.62
2019-08-05,-0.06,-0.07,0.88
2007-03-02,-0.05,-0.07,0.74
2020-02-27,-0.05,-0.06,0.81
2007-02-28,-0.05,-0.06,0.74
2018-10-10,-0.05,-0.06,0.76
2018-10-15,-0.03,-0.06,0.53
